# Results Table and Experiments

This notebook produces the **main results table** (every model scored on training and test users) and the **experiments** (time windows and feature families). Every score comes from the shared scoring functions in Section 2, so all models are compared the same way.

The main model is the **BST** (`Test_BST_23_Features.ipynb`). **LightGBM** (`Jingxuan_LightGBM_23_Features.ipynb`) is the white-box comparison model, trained on the same 23 features. The experiments test two hypotheses on validation users: **(1)** history older than 7 days improves the ranking (7.1), and **(2)** the re-ranker's value comes from the user's own history and from popularity rather than from matching the query (7.2 to 7.5). The feature-family experiment is run on both models, BST first. The time-window experiment and feature importance use LightGBM; Section 7 says why.

## Agenda
1. Setup
2. Shared scoring code
3. Loading helpers
4. Cold-product flag
5. Baselines: Random, Popularity, NMF, NCF
6. BST and LightGBM rows of the results table
    - 6.1 Sensitivity to the relevance weights
7. Experiments (validation users)
    - 7.1 Time-window experiment (LightGBM, features from `Elysia_Window_Features.ipynb`)
    - 7.2 Feature-family experiment on BST (runs from Step 10 of `Test_BST_23_Features.ipynb`)
    - 7.3 Feature-family experiment on LightGBM
    - 7.4 All personal-history features removed at once (LightGBM)
    - 7.5 Feature importance (LightGBM)
8. Save everything

## Inputs (all under `BT4222 Final/JDSearch/`, read only)
| File | Produced by | Used for |
|---|---|---|
| `processed/lgbm_features_all/` (`{train,val,test}.parquet`, `{split}_groups.npy`, `feature_dictionary_part3.csv`) | feature pipeline of `Part3_Feature_Engineering.ipynb`, on the split from `Siqian_Data_Preprocessing.ipynb` | features, labels, group sizes, feature families |
| `processed/lgbm_features_window_7d/` and `..._90d/` | `Elysia_Window_Features.ipynb` | features from 7 and 90 days of history (7.1) |
| `model_artifacts/nmf_wh.pkl`, `nmf_artifacts.pkl`, `ncf_model.pt` | `ZhangNingJing_cleaned.ipynb` | trained NMF and NCF models |
| `processed/ningjing_candidate_features_checkpoint.parquet` | `ZhangNingJing_cleaned.ipynb` | `item_pop_train`, for the check in 5.3 only |
| `model_artifacts/lightgbm_popularity_query_search/cached_merged_features/selected_features/` | `Jingxuan_LightGBM_23_Features.ipynb` | the seven query-match features |
| `model_artifacts/lightgbm_popularity_query_search/full/{train,test}_predictions.parquet` | `Jingxuan_LightGBM_23_Features.ipynb` | LightGBM predictions |
| `processed/bst_literal/{train,test}_predictions.parquet` | `Test_BST_23_Features.ipynb` (Step 9 for train) | BST predictions |
| `processed/bst_experiments/` | `Test_BST_23_Features.ipynb`, Step 10 | BST family-experiment predictions (7.2) |
| `Raw Data/user_behavior_data.txt` | raw data | cold flag |

**Outputs** go to `model_artifacts/4B_results/`: `main_results_table.csv`, `all_results.csv`, `nmf_ncf_scores_{train,test}.parquet`, `window_experiment.csv`, `bst_family_experiment.csv`, `family_experiment_runs.csv`, `family_experiment.csv`, `family_full_model.txt`, `feature_importance.csv`.

The input files call the user `session_id` and the product `wid` (`item_id` in the ZhangNingJing files); the scoring code uses `user_id` and `item_id`, and `to_pred()` in Section 3 renames them.

## 1. Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import gc, json
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

BASE_DIR = Path("/content/drive/MyDrive/BT4222 Final")
PROCESSED = BASE_DIR / "JDSearch" / "processed"
FEAT_DIR = PROCESSED / "lgbm_features_all"                               # full-year features (Part3_Feature_Engineering pipeline)
P2_FILE = PROCESSED / "ningjing_candidate_features_checkpoint.parquet"  # ZhangNingJing_cleaned checkpoint
USER_TXT = BASE_DIR / "JDSearch" / "Raw Data" / "user_behavior_data.txt"
RESULTS_DIR = BASE_DIR / "JDSearch" / "model_artifacts" / "4B_results"   # outputs of this notebook; processed/ is read-only
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

## 2. Shared scoring code

Every model (Random, Popularity, NMF, NCF, BST, LightGBM) is scored with these functions, so the numbers are comparable. The same functions are saved as `Scoring_Code.ipynb` for the other notebooks. Each metric is computed per user and averaged over users.

**Input `pred_df`:** one row per (user, candidate) with `user_id`, `item_id`, `label` (0 none, 1 click, 2 cart, 3 purchase), `score` (higher = ranked higher, any scale) and optionally `is_cold`. All of a user's candidates are kept, because NDCG needs the full list for the ideal ranking.

**Metrics:** NDCG@10 (main metric, gain $2^{label}-1$: click 1, cart 3, purchase 7), MRR (rank of the first engaged product), P@1 (whether the top product was engaged), EngRecall@K and PurRecall@K (share of engaged or purchased products in the top K, purchase recall averaged only over users who purchased), and EngRecall@K split into cold and warm products.

**Ties are broken randomly** with a fixed seed. The raw data stores engaged products first in every candidate list, so if tied products kept their stored order a model with no information would score about 0.94 instead of about 0.08 (shown in 2.5).

In [3]:
RANDOM_SEED = 4222
LABEL_NAMES = {0: 'no interaction', 1: 'click', 2: 'add to cart', 3: 'purchase'}

### 2.1 Per-user metric functions
Each function takes the labels of one user's candidates, already sorted in the model's ranked order (best first).

In [4]:
def dcg_at_k(relevance_scores, k):
    # DCG with exponential gain 2^label - 1 and log2 position discount
    relevance_scores = np.asarray(relevance_scores, dtype=float)[:k]
    if relevance_scores.size:
        return np.sum((2**relevance_scores - 1) / np.log2(np.arange(2, relevance_scores.size + 2)))
    return 0.0


def ndcg_at_k(relevance_scores, k):
    # IMPORTANT: pass the user's FULL ranked list (not only the top k),
    # so that the ideal ranking is built from all of the user's candidates
    dcg = dcg_at_k(relevance_scores, k)
    ideal_scores = sorted(relevance_scores, reverse=True)
    idcg = dcg_at_k(ideal_scores, k)
    return dcg / idcg if idcg > 0 else 0.0


def reciprocal_rank(relevance_scores, min_label=1):
    # 1 / (position of the first relevant product). Rank 1 -> 1.0, rank 4 -> 0.25. 0 if none.
    hits = np.flatnonzero(np.asarray(relevance_scores) >= min_label)
    return 1.0 / (hits[0] + 1) if hits.size else 0.0


def precision_at_k(relevance_scores, k, min_label=1):
    # share of the top k products that are relevant (k=1 gives P@1)
    top_k = np.asarray(relevance_scores)[:k]
    return float(np.sum(top_k >= min_label)) / k


def recall_at_k(relevance_scores, k, min_label=1, max_label=None, mask=None):
    """
    Share of the user's relevant products that appear in the top k.
    min_label / max_label: which labels count as relevant (engagement: min_label=1, purchase: min_label=3)
    mask: optional boolean array (same order as relevance_scores); only products where mask is True are counted
    Returns None if the user has no relevant product (such users are left out of the average, not counted as 0).
    """
    rel = np.asarray(relevance_scores)
    relevant = rel >= min_label
    if max_label is not None:
        relevant &= rel <= max_label
    if mask is not None:
        relevant &= np.asarray(mask, dtype=bool)
    m = relevant.sum()
    if m == 0:
        return None
    return float(relevant[:k].sum()) / m

### 2.2 Ranking and scoring all users
`evaluate_ranking()` is the main function. It checks the input, ranks each user's candidates (random tie-breaking), then computes every metric for each user and averages over users.

In [5]:
def rank_candidates(pred_df, seed=RANDOM_SEED):
    # Sort every user's candidates by score (highest first), breaking ties with a random number
    # so the stored order is never used. Adds a `rank` column (1 = top).
    rng = np.random.default_rng(seed)
    df = pred_df.copy()
    df['_tie_break'] = rng.random(len(df))
    df = df.sort_values(['user_id', 'score', '_tie_break'],
                        ascending=[True, False, True], kind='mergesort')
    df['rank'] = df.groupby('user_id', sort=False).cumcount() + 1
    return df.drop(columns='_tie_break').reset_index(drop=True)


def check_predictions(pred_df):
    # Quick sanity checks before scoring. Prints warnings, never stops the run.
    needed = {'user_id', 'item_id', 'label', 'score'}
    missing = needed - set(pred_df.columns)
    assert not missing, f"Missing columns: {missing}"

    if pred_df['score'].isna().any():
        print(f"WARNING: {pred_df['score'].isna().sum()} NaN scores (they will be ranked last)")
    if not pred_df['label'].isin([0, 1, 2, 3]).all():
        print("WARNING: labels outside 0-3 found")
    if 'is_cold' in pred_df.columns and pred_df['is_cold'].isna().any():
        print("WARNING: is_cold has missing values (they will be counted as cold)")
    n_dup = pred_df.duplicated(['user_id', 'item_id']).sum()
    if n_dup:
        print(f"WARNING: {n_dup} duplicated (user_id, item_id) rows")

    per_user = pred_df.groupby('user_id')['label'].agg(['max', 'min'])
    no_pos = (per_user['max'] == 0).sum()
    no_neg = (per_user['min'] > 0).sum()
    if no_pos or no_neg:
        print(f"WARNING: {no_pos} users with no engaged product and {no_neg} users with "
              f"no non-engaged product. These should have been removed in cleaning "
              f"(interim report: 2,103 + 553 excluded users).")

    if pred_df.groupby('score').ngroups < 0.5 * len(pred_df):
        print("NOTE: many tied scores. Ties are broken randomly (see Section 2).")


def evaluate_ranking(pred_df, ndcg_k=10, k_list=(5, 10, 20), seed=RANDOM_SEED,
                     per_user=False, verbose=True):
    """
    Score a model's predictions.
    pred_df: DataFrame with user_id, item_id, label, score (+ optional is_cold)
    ndcg_k: cut-off for NDCG (10 in the interim report)
    k_list: cut-offs for the Recall@K metrics
    per_user: if True, also return a per-user table (for diagnostics)
    Returns a dict of macro-averaged metrics (and the per-user DataFrame if per_user=True).
    """
    if verbose:
        check_predictions(pred_df)

    df = pred_df.copy()
    df['score'] = df['score'].fillna(-np.inf)
    df = rank_candidates(df, seed=seed)

    has_cold = 'is_cold' in df.columns
    users = df['user_id'].to_numpy()
    labels = df['label'].to_numpy()
    cold = df['is_cold'].to_numpy(dtype=bool) if has_cold else None

    # start index of each user's block (df is sorted by user)
    starts = np.flatnonzero(np.r_[True, users[1:] != users[:-1]])
    ends = np.r_[starts[1:], len(df)]

    rows = []
    for s, e in zip(starts, ends):
        rel = labels[s:e]                     # this user's labels in ranked order
        row = {'user_id': users[s],
               'n_candidates': e - s,
               f'NDCG@{ndcg_k}': ndcg_at_k(rel, ndcg_k),
               'MRR': reciprocal_rank(rel),
               'P@1': precision_at_k(rel, 1)}
        for k in k_list:
            row[f'EngRecall@{k}'] = recall_at_k(rel, k, min_label=1)
            row[f'PurRecall@{k}'] = recall_at_k(rel, k, min_label=3)
            if has_cold:
                c = cold[s:e]
                row[f'EngRecall@{k} cold'] = recall_at_k(rel, k, min_label=1, mask=c)
                row[f'EngRecall@{k} warm'] = recall_at_k(rel, k, min_label=1, mask=~c)
        rows.append(row)

    user_df = pd.DataFrame(rows)
    metric_cols = [c for c in user_df.columns if c not in ('user_id', 'n_candidates')]

    # macro-average: None (user has no relevant product of that kind) is skipped
    results = {col: float(user_df[col].dropna().mean()) for col in metric_cols}
    results['n_users'] = len(user_df)
    results['n_users_with_purchase'] = int(user_df[f'PurRecall@{k_list[0]}'].notna().sum())
    if has_cold:
        results['n_users_with_cold_engaged'] = int(user_df[f'EngRecall@{k_list[0]} cold'].notna().sum())

    if verbose:
        print(f"Scored {results['n_users']:,} users "
              f"({results['n_users_with_purchase']:,} with a purchase)")

    if per_user:
        return results, user_df
    return results

### 2.3 Reference rankings
A perfect ranking (score = true label) must give 1.0. A random ranking is the floor any real model must beat.

In [6]:
def random_scores(pred_df, seed=RANDOM_SEED):
    # random ranking: any real model must beat this
    rng = np.random.default_rng(seed + 1)
    return rng.random(len(pred_df))


def oracle_scores(pred_df):
    # perfect ranking (score = true label): NDCG, MRR and P@1 must all equal 1.0
    return pred_df['label'].to_numpy(dtype=float)

### 2.4 Results table helpers
Collect each model's scores with `add_result`, then `results_table` builds one table with a column per (metric, split).

In [7]:
MAIN_COLUMNS = ['NDCG@10', 'MRR', 'P@1', 'EngRecall@10', 'PurRecall@10',
                'EngRecall@10 cold', 'EngRecall@10 warm']


def add_result(results_list, model_name, split_name, metrics):
    # Append one model's scores to the shared results list.
    row = {'model': model_name, 'split': split_name}
    row.update(metrics)
    results_list.append(row)


def results_table(results_list, columns=MAIN_COLUMNS, decimals=4):
    # One row per model, one column per (metric, split).
    df = pd.DataFrame(results_list)
    cols = [c for c in columns if c in df.columns]
    table = df.pivot_table(index='model', columns='split', values=cols, sort=False)
    table = table.reindex(columns=cols, level=0)
    return table.round(decimals)

### 2.5 Tests
**Hand example:** labels in ranked order `[0, 3, 0, 1, 0]`: the purchase is at rank 2 and the click at rank 4. DCG@10 $= \frac{7}{\log_2 3} + \frac{1}{\log_2 5} = 4.848$, ideal order `[3, 1, 0, 0, 0]` gives IDCG@10 $= 7 + \frac{1}{\log_2 3} = 7.631$, so NDCG@10 $= 0.635$. MRR = 1/2, P@1 = 0, Engagement Recall@2 = 1/2, Purchase Recall@2 = 1.

**Tie-breaking check:** fake lists with the JDSearch layout (engaged products stored first) and every score tied. The correct answer is "no better than random".

**Perfect and random rankings:** perfect must give 1.0 on every metric. Random is the floor.

In [8]:
ranked_labels = [0, 3, 0, 1, 0]
print('NDCG@10     :', round(ndcg_at_k(ranked_labels, 10), 3), '(expect 0.635)')
print('MRR         :', reciprocal_rank(ranked_labels), '(expect 0.5)')
print('P@1         :', precision_at_k(ranked_labels, 1), '(expect 0.0)')
print('EngRecall@2 :', recall_at_k(ranked_labels, 2, min_label=1), '(expect 0.5)')
print('PurRecall@2 :', recall_at_k(ranked_labels, 2, min_label=3), '(expect 1.0)')
print('PurRecall when the user bought nothing:', recall_at_k([0, 1, 0], 2, min_label=3), '(expect None, user left out of the average)')

rng = np.random.default_rng(0)
rows = []
for u in range(1000):
    n = rng.integers(20, 150)
    label = np.zeros(n, dtype=int)
    n_pos = rng.integers(1, 4)
    label[:n_pos] = rng.choice([1, 1, 1, 2, 3], n_pos)      # engaged products stored at the top
    rows.append(pd.DataFrame({'user_id': u, 'item_id': np.arange(n), 'label': label,
                              'is_cold': rng.random(n) < 0.8}))
fake = pd.concat(rows, ignore_index=True)
fake['score'] = 0.0

wrong = fake.sort_values(['user_id', 'score'], ascending=[True, False], kind='mergesort')
print('\nNDCG@10 if ties keep the stored order :',
      round(np.mean([ndcg_at_k(g['label'].values, 10) for _, g in wrong.groupby('user_id')]), 3), '(fake, far too high)')
print('NDCG@10 with random tie-breaking      :', round(evaluate_ranking(fake, verbose=False)['NDCG@10'], 3), '(random level)')

fake['score'] = oracle_scores(fake)
print('\nPerfect:', {k: round(v, 3) for k, v in evaluate_ranking(fake, verbose=False).items() if k in MAIN_COLUMNS})
fake['score'] = random_scores(fake)
print('Random :', {k: round(v, 3) for k, v in evaluate_ranking(fake, verbose=False).items() if k in MAIN_COLUMNS})
del fake, wrong, rows, rng, ranked_labels

NDCG@10     : 0.635 (expect 0.635)
MRR         : 0.5 (expect 0.5)
P@1         : 0.0 (expect 0.0)
EngRecall@2 : 0.5 (expect 0.5)
PurRecall@2 : 1.0 (expect 1.0)
PurRecall when the user bought nothing: None (expect None, user left out of the average)

NDCG@10 if ties keep the stored order : 0.941 (fake, far too high)
NDCG@10 with random tie-breaking      : 0.076 (random level)

Perfect: {'NDCG@10': 1.0, 'MRR': 1.0, 'P@1': 1.0, 'EngRecall@10': 1.0, 'PurRecall@10': 1.0, 'EngRecall@10 cold': 1.0, 'EngRecall@10 warm': 1.0}
Random : {'NDCG@10': 0.082, 'MRR': 0.115, 'P@1': 0.034, 'EngRecall@10': 0.16, 'PurRecall@10': 0.153, 'EngRecall@10 cold': 0.162, 'EngRecall@10 warm': 0.167}


## 3. Loading helpers

`load_split()` reads one `lgbm_features_all` split file, only the columns asked for, keeping whole sessions together. The seven query-match features are not in those files, so it merges them in from the `Jingxuan_LightGBM_23_Features` feature cache. `to_pred()` converts any table plus a score column into the scoring-code format.

In [9]:
ALL_FEATURES = json.loads((FEAT_DIR / "feature_list.json").read_text())
FAMILY_OF = pd.read_csv(FEAT_DIR / "feature_dictionary_part3.csv").set_index("feature")["family"].to_dict()

FAMILY_NAMES = {
    "context": "User activity by time window",
    "A": "Exact product history",
    "B": "Brand / shop / category affinity",
    "C": "Title similarity to history",
    "D": "item2vec / term2vec similarity",
    "E": "Similarity to the last action",
    "F": "History under the same query",
    "G": "Popularity priors (training users only)",
    "H": "Relative to the other candidates in the slate",
    "Q": "Current-query match",
}

# The query-match columns are not in lgbm_features_all. load_split() takes them from the feature cache of
# Jingxuan_LightGBM_23_Features, which has the corrected name_query_match and the recomputed query_match_count.
LGBM_ROOT = BASE_DIR / "JDSearch" / "model_artifacts" / "lightgbm_popularity_query_search"   # output folder of Jingxuan_LightGBM_23_Features
QUERY_CACHE = LGBM_ROOT / "cached_merged_features" / "selected_features"
QUERY_FEATURES = ["name_query_match", "brand_name_query_match", "cate_name_1_query_match",
                  "cate_name_2_query_match", "cate_name_3_query_match", "cate_name_4_query_match",
                  "query_match_count"]
FAMILY_OF.update({f: "Q" for f in QUERY_FEATURES})


def load_split(split, feats=(), folder=FEAT_DIR, session_fraction=1.0, seed=42):
    # Read session_id, wid, label and the chosen features of one split.
    # session_fraction < 1 keeps a random subset of WHOLE sessions (for faster training).
    group = np.load(folder / f"{split}_groups.npy")
    keep_sess = np.ones(len(group), bool)
    if session_fraction < 1.0:
        keep_sess = np.random.default_rng(seed).random(len(group)) < session_fraction
    keep_row = np.repeat(keep_sess, group)

    q_feats = [f for f in feats if f in QUERY_FEATURES]
    base_feats = [f for f in feats if f not in QUERY_FEATURES]
    parts, pos = [], 0
    pf = pq.ParquetFile(folder / f"{split}.parquet")
    for batch in pf.iter_batches(batch_size=500_000, columns=["session_id", "wid", "label"] + base_feats):
        b = batch.to_pandas()
        m = keep_row[pos:pos + len(b)]
        pos += len(b)
        b = b[m]
        for f in base_feats:
            b[f] = b[f].astype(np.float32)
        parts.append(b)
    df = pd.concat(parts, ignore_index=True)
    assert pos == len(keep_row), "group file does not match the data file"

    if q_feats:   # add the query-match columns from the LightGBM feature cache, keeping the row order of lgbm_features_all
        assert (QUERY_CACHE / f"{split}.parquet").exists(), "query-match cache missing: Section 4 of Jingxuan_LightGBM_23_Features writes it"
        q = pd.read_parquet(QUERY_CACHE / f"{split}.parquet", columns=["session_id", "wid"] + q_feats)
        q = q.drop_duplicates(["session_id", "wid"])
        for f in q_feats:
            q[f] = q[f].astype(np.float32)
        df = df.merge(q, on=["session_id", "wid"], how="left", validate="many_to_one")
        assert df[q_feats].notna().all().all(), "some candidates have no query-match values"
        del q; gc.collect()
    df = df[["session_id", "wid", "label"] + list(feats)]
    print(f"{split}: {df['session_id'].nunique():,} sessions, {len(df):,} rows, {len(feats)} features")
    return df, group[keep_sess]


def to_pred(df, score):
    # Scoring-code format: user_id, item_id, label, score (+ is_cold if present)
    pred = pd.DataFrame({"user_id": df["session_id"].to_numpy(),
                         "item_id": df["wid"].to_numpy(),
                         "label": df["label"].to_numpy(),
                         "score": np.asarray(score, dtype=np.float64)})
    if "is_cold" in df.columns:
        pred["is_cold"] = df["is_cold"].to_numpy()
    return pred

## 4. Cold-product flag

A **cold product** never appears in any user's history (about 80% of unique candidate products). Only history is used, not labels. This differs from the popularity buckets in `NingJing_Bias_Diagnosis.ipynb`, which group products by engagement among training users.

In [10]:
hist_col = pd.read_csv(USER_TXT, sep="\t", dtype=str, usecols=["history_wid_list"], keep_default_na=False)
warm_items = set()
for s in hist_col["history_wid_list"]:
    if s:
        warm_items.update(int(x) for x in s.split("_"))
del hist_col; gc.collect()
print(f"products appearing in some history: {len(warm_items):,}")


def add_cold(df):
    df["is_cold"] = ~df["wid"].astype("int64").isin(warm_items)
    return df

products appearing in some history: 5,584,543


## 5. Baselines

| Baseline | Score |
|---|---|
| **Random** | random number |
| **Popularity** | `item_ctr_train`: smoothed engagement rate among training sessions, out-of-fold for training rows so no row's own label is in its score |
| **NMF** / **NCF** | the NMF or NCF score from `ZhangNingJing_cleaned.ipynb` where the product has a vector (about 22% of candidates), otherwise the popularity score |

**5.1** recomputes the NMF and NCF scores from the saved models (`nmf_wh.pkl`, `nmf_artifacts.pkl`, `ncf_model.pt`) with the same calculation as that notebook, since the saved checkpoint does not include them. Nothing is retrained. **5.2** scores the four baselines on training and test users.

These numbers are lower than the interim report's because the interim fallback (`pop_score`) counted engagement over all sessions including test sessions, which leaked the labels, and because our NDCG uses graded gains. **5.3** checks that the two scoring codes agree when given the same inputs.

### 5.1 Recompute the NMF and NCF scores

Loads the saved NMF factors (`nmf_wh.pkl`, `nmf_artifacts.pkl`) and the saved NCF weights (`ncf_model.pt`) and scores every candidate row with the same calculation as `ZhangNingJing_cleaned.ipynb`. Nothing is retrained. Products with no vector (about 78% of candidates) fall back to `item_ctr_train`, the training-only popularity score. The scores are written to `4B_results/nmf_ncf_scores_{train,test}.parquet` once. When those files already exist the cell skips the recompute and only reports that they are ready.

In [ ]:
# 5.1 Recompute the NMF and NCF scores from the models saved by ZhangNingJing_cleaned.ipynb
import pickle, torch, torch.nn as nn
ART = BASE_DIR / "JDSearch" / "model_artifacts"

class NCF(nn.Module):          # same architecture as in ZhangNingJing_cleaned.ipynb
    def __init__(self, n_users, n_items, emb_dim=16, hidden_dims=(32, 16)):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, emb_dim)
        self.item_emb = nn.Embedding(n_items, emb_dim)
        layers, input_dim = [], emb_dim * 2
        for h in hidden_dims:
            layers += [nn.Linear(input_dim, h), nn.ReLU()]
            input_dim = h
        layers.append(nn.Linear(input_dim, 1))
        self.mlp = nn.Sequential(*layers)
    def forward(self, user, item):
        x = torch.cat([self.user_emb(user), self.item_emb(item)], dim=1)
        return self.mlp(x).squeeze(-1)

if not all((RESULTS_DIR / f"nmf_ncf_scores_{s}.parquet").exists() for s in ["train", "test"]):
    with open(ART / "nmf_wh.pkl", "rb") as f:
        wh = pickle.load(f)
    with open(ART / "nmf_artifacts.pkl", "rb") as f:
        art = pickle.load(f)
    W, H = wh["W"].astype(np.float32), wh["H"].astype(np.float32)       # user factors, item factors
    col_of = pd.Series(art["item_id_to_col"])                            # product id -> column in H
    col_of.index = col_of.index.astype("int64")
    ncf = NCF(W.shape[0], H.shape[1], emb_dim=16)
    ncf.load_state_dict(torch.load(ART / "ncf_model.pt", map_location="cpu"))
    ncf.eval()

    for split in ["train", "test"]:
        df, _ = load_split(split)                                        # session_id, wid, label
        idx = col_of.reindex(df["wid"].to_numpy()).to_numpy()           # NaN if the product has no vector
        has = ~np.isnan(idx)
        sid = df["session_id"].to_numpy()[has].astype(np.int64)         # session_id = row in user file = row in W
        iid = idx[has].astype(np.int64)
        nmf_s = np.full(len(df), np.nan, np.float32)
        ncf_s = np.full(len(df), np.nan, np.float32)
        nmf_v, ncf_v = np.empty(len(sid), np.float32), np.empty(len(sid), np.float32)
        for a in range(0, len(sid), 500_000):
            b = a + 500_000
            nmf_v[a:b] = (W[sid[a:b]] * H[:, iid[a:b]].T).sum(axis=1)       # dot product, as in ZhangNingJing_cleaned.ipynb
            with torch.no_grad():
                ncf_v[a:b] = torch.sigmoid(ncf(torch.from_numpy(sid[a:b]), torch.from_numpy(iid[a:b]))).numpy()
        nmf_s[has], ncf_s[has] = nmf_v, ncf_v
        pd.DataFrame({"session_id": df["session_id"], "wid": df["wid"], "nmf_score": nmf_s, "ncf_score": ncf_s}) \
          .to_parquet(RESULTS_DIR / f"nmf_ncf_scores_{split}.parquet", index=False)
        print(f"{split}: {has.mean():.1%} of candidates have an NMF/NCF vector (ZhangNingJing_cleaned reported 21.9%)")
        del df, nmf_s, ncf_s, nmf_v, ncf_v; gc.collect()
    del W, H, ncf; gc.collect()
print("NMF/NCF scores ready in", RESULTS_DIR)

NMF/NCF scores ready in /content/drive/MyDrive/BT4222 Final/JDSearch/model_artifacts/4B_results


### 5.2 Score the four baselines

Random, Popularity, NMF and NCF scored on training and test users with the shared scoring code. Each result is added to `all_results`, which the BST and LightGBM rows join in Section 6.

In [11]:
# 5.2 Score the four baselines on training and test users
all_results = []
per_split = {}
OFFSET = 1e6   # products with a vector always rank above the popularity fallback (as in ZhangNingJing_cleaned.ipynb)

for split in ["train", "test"]:
    df, _ = load_split(split, ["item_ctr_train"])
    df = add_cold(df)
    sc = pd.read_parquet(RESULTS_DIR / f"nmf_ncf_scores_{split}.parquet").drop_duplicates(["session_id", "wid"])
    df = df.merge(sc, on=["session_id", "wid"], how="left", validate="many_to_one")
    del sc; gc.collect()

    scores = {
        "Random": random_scores(df),
        "Popularity (training users)": df["item_ctr_train"],
        "NMF (+ popularity fallback)": np.where(df["nmf_score"].notna(), df["nmf_score"] + OFFSET, df["item_ctr_train"]),
        "NCF (+ popularity fallback)": np.where(df["ncf_score"].notna(), df["ncf_score"] + OFFSET, df["item_ctr_train"]),
    }
    for name, s in scores.items():
        print(f"--- {name} ({split})")
        add_result(all_results, name, split, evaluate_ranking(to_pred(df, s)))
    per_split[split] = df[["session_id", "wid", "label", "is_cold"]]
    del df; gc.collect()

results_table(all_results)

train: 136,940 sessions, 12,279,873 rows, 1 features
--- Random (train)
Scored 136,940 users (16,289 with a purchase)
--- Popularity (training users) (train)
NOTE: many tied scores. Ties are broken randomly (see Section 2).
Scored 136,940 users (16,289 with a purchase)
--- NMF (+ popularity fallback) (train)
NOTE: many tied scores. Ties are broken randomly (see Section 2).
Scored 136,940 users (16,289 with a purchase)
--- NCF (+ popularity fallback) (train)
NOTE: many tied scores. Ties are broken randomly (see Section 2).
Scored 136,940 users (16,289 with a purchase)
test: 17,118 sessions, 1,531,525 rows, 1 features
--- Random (test)
Scored 17,118 users (1,988 with a purchase)
--- Popularity (training users) (test)
NOTE: many tied scores. Ties are broken randomly (see Section 2).
Scored 17,118 users (1,988 with a purchase)
--- NMF (+ popularity fallback) (test)
NOTE: many tied scores. Ties are broken randomly (see Section 2).
Scored 17,118 users (1,988 with a purchase)
--- NCF (+ popul

NDCG@10             MRR             P@1          \
split                         train    test   train    test   train    test   
model                                                                         
Random                       0.0804  0.0815  0.0987  0.0989  0.0277  0.0274   
Popularity (training users)  0.2476  0.2549  0.2613  0.2677  0.1539  0.1577   
NMF (+ popularity fallback)  0.3053  0.3043  0.3067  0.3047  0.1752  0.1752   
NCF (+ popularity fallback)  0.2996  0.3004  0.2990  0.2982  0.1648  0.1653   

                            EngRecall@10         PurRecall@10          \
split                              train    test        train    test   
model                                                                   
Random                            0.1565  0.1586       0.1365  0.1369   
Popularity (training users)       0.3731  0.3853       0.3841  0.4018   
NMF (+ popularity fallback)       0.4766  0.4752       0.5352  0.5487   
NCF (+ popularity fallback)       0.4759  0.4777       0.5307  0.5401   

                            EngRecall@10 cold         EngRecall@10 warm  \
split                                   train    test             train   
model                                                                     
Random                                 0.1659  0.1792            0.1476   
Popularity (training users)            0.1490  0.1586            0.4136   
NMF (+ popularity fallback)            0.1175  0.1247            0.5517   
NCF (+ popularity fallback)            0.1175  0.1247            0.5511   

                                     
split                          test  
model                                
Random                       0.1455  
Popularity (training users)  0.4286  
NMF (+ popularity fallback)  0.5506  
NCF (+ popularity fallback)  0.5539

**Result:** Popularity alone (NDCG@10 0.255 on test) is about three times Random (0.082), and NMF (0.304) and NCF (0.300) add about 0.05 on top of it. Since both fall back to the popularity score for the 78% of candidates without a vector, most of their score is the popularity baseline's.

### 5.3 Check: do the old and new scoring codes agree?

With binary labels and the interim popularity score (`item_pop_train`), our scoring code should reproduce the interim test numbers, **NDCG@10 0.2820, MRR 0.2907, P@1 0.1732**, apart from the tie-breaking seed.

In [12]:
chk = pd.read_parquet(P2_FILE, columns=["session_id", "item_id", "label", "item_pop_train"],
                      filters=[("split", "==", "test")])
chk_pred = pd.DataFrame({"user_id": chk["session_id"], "item_id": chk["item_id"],
                         "label": (chk["label"] > 0).astype(int),      # binary relevance, as in the interim scoring
                         "score": chk["item_pop_train"]})
m = evaluate_ranking(chk_pred, verbose=False)
print(f"Our code, binary labels: NDCG@10 {m['NDCG@10']:.4f}  MRR {m['MRR']:.4f}  P@1 {m['P@1']:.4f}")
print("Interim report:          NDCG@10 0.2820  MRR 0.2907  P@1 0.1732")
del chk, chk_pred
_ = gc.collect()

Our code, binary labels: NDCG@10 0.2816  MRR 0.2906  P@1 0.1726
Interim report:          NDCG@10 0.2820  MRR 0.2907  P@1 0.1732


The two codes agree to three decimal places, so the lower baseline numbers in 5.2 come from the leakage correction and the graded relevance, not from the scoring code.

## 6. BST and LightGBM rows of the results table

The two Section 2 models of the report, both trained on the same 23 features and the same split:
- **BST** (`Test_BST_23_Features.ipynb`): the main model, a Behaviour Sequence Transformer. Predictions in `processed/bst_literal/`.
- **LightGBM** (`Jingxuan_LightGBM_23_Features.ipynb`): the white-box comparison model. Predictions in `model_artifacts/lightgbm_popularity_query_search/full/`.

Each notebook saved `train_predictions.parquet` and `test_predictions.parquet` (`session_id`, `wid`, `label`, `score`). They are scored here with the same functions as the baselines and added to the results table, with the cold flag from Section 4.

In [13]:
BST_DIR = PROCESSED / "bst_literal"
LGBM_DIR = LGBM_ROOT / "full"
MODELS = {
    # 'name in the table': {'train': 'file', 'test': 'file'}
    "BST":      {"train": BST_DIR / "train_predictions.parquet",
                 "test":  BST_DIR / "test_predictions.parquet"},
    "LightGBM": {"train": LGBM_DIR / "train_predictions.parquet",
                 "test":  LGBM_DIR / "test_predictions.parquet"},
}

for name, files in MODELS.items():
    for split, f in files.items():
        if not Path(f).exists():
            print("not ready yet:", f); continue
        p = pd.read_parquet(f)
        p = p.rename(columns={"session_id": "user_id", "wid": "item_id"})
        p = p.merge(per_split[split].rename(columns={"session_id": "user_id", "wid": "item_id"})[["user_id", "item_id", "is_cold"]]
                     .drop_duplicates(["user_id", "item_id"]), on=["user_id", "item_id"], how="left")
        print(f"--- {name} ({split})")
        add_result(all_results, name, split, evaluate_ranking(p))

results_table(all_results)

--- BST (train)
NOTE: many tied scores. Ties are broken randomly (see Section 2).
Scored 136,940 users (16,289 with a purchase)
--- BST (test)
NOTE: many tied scores. Ties are broken randomly (see Section 2).
Scored 17,118 users (1,988 with a purchase)
--- LightGBM (train)
Scored 136,940 users (16,289 with a purchase)
--- LightGBM (test)
Scored 17,118 users (1,988 with a purchase)


NDCG@10             MRR             P@1          \
split                         train    test   train    test   train    test   
model                                                                         
Random                       0.0804  0.0815  0.0987  0.0989  0.0277  0.0274   
Popularity (training users)  0.2476  0.2549  0.2613  0.2677  0.1539  0.1577   
NMF (+ popularity fallback)  0.3053  0.3043  0.3067  0.3047  0.1752  0.1752   
NCF (+ popularity fallback)  0.2996  0.3004  0.2990  0.2982  0.1648  0.1653   
BST                          0.4395  0.3848  0.4432  0.3943  0.2967  0.2578   
LightGBM                     0.4137  0.4054  0.4210  0.4122  0.2827  0.2718   

                            EngRecall@10         PurRecall@10          \
split                              train    test        train    test   
model                                                                   
Random                            0.1565  0.1586       0.1365  0.1369   
Popularity (training users)       0.3731  0.3853       0.3841  0.4018   
NMF (+ popularity fallback)       0.4766  0.4752       0.5352  0.5487   
NCF (+ popularity fallback)       0.4759  0.4777       0.5307  0.5401   
BST                               0.6326  0.5610       0.7236  0.6034   
LightGBM                          0.5953  0.5890       0.6431  0.6412   

                            EngRecall@10 cold         EngRecall@10 warm  \
split                                   train    test             train   
model                                                                     
Random                                 0.1659  0.1792            0.1476   
Popularity (training users)            0.1490  0.1586            0.4136   
NMF (+ popularity fallback)            0.1175  0.1247            0.5517   
NCF (+ popularity fallback)            0.1175  0.1247            0.5511   
BST                                    0.2657  0.1885            0.7099   
LightGBM                               0.2020  0.1988            0.6820   

                                     
split                          test  
model                                
Random                       0.1455  
Popularity (training users)  0.4286  
NMF (+ popularity fallback)  0.5506  
NCF (+ popularity fallback)  0.5539  
BST                          0.6466  
LightGBM                     0.6803

**Result:** Both feature-based models beat every baseline by a wide margin: LightGBM 0.405 and BST 0.385 test NDCG@10 against 0.304 for the best baseline. LightGBM is ahead of BST on every metric, by 0.020 on NDCG@10. BST scores 0.440 on training users against 0.385 on test, a gap of 0.055, while LightGBM's gap is 0.008, so the BST fits the training users' histories more closely than it generalises. Scores on training users are shown for this comparison only; the test columns are the reported results.

### 6.1 Sensitivity to the relevance weights

Test NDCG@10 with the graded gains used throughout (click 1, cart 3, purchase 7) next to binary relevance (every engaged product counts 1), to show whether the choice of weights changes the scores or the order of the two models.

In [14]:
sens = []
for name, files in MODELS.items():
    p = pd.read_parquet(files["test"]).rename(columns={"session_id": "user_id", "wid": "item_id"})
    graded = evaluate_ranking(p, verbose=False)["NDCG@10"]
    p["label"] = (p["label"] > 0).astype(int)
    binary = evaluate_ranking(p, verbose=False)["NDCG@10"]
    sens.append({"model": name, "NDCG@10 graded (test)": round(graded, 4), "NDCG@10 binary (test)": round(binary, 4)})
pd.DataFrame(sens)

,model,NDCG@10 graded (test),NDCG@10 binary (test)
0,BST,0.3848,0.3896
1,LightGBM,0.4054,0.4104


**Result:** Binary relevance raises both models by 0.005 (LightGBM 0.410, BST 0.390) and leaves the order unchanged, so the conclusions do not depend on the choice of gains.

## 7. Experiments

All experiments are scored on **validation** users, so the test users stay untouched for the results table. They test two hypotheses:

- **Hypothesis 1 (7.1):** Including user history older than 7 days improves ranking quality.
    - JD kept clicks for about 7 days but carts for 90 days and orders for 365, so features can be built from the last week of history or from up to a year of it. Hypothesis 1 is supported if the models built from 90 days and from a full year of history score higher than the model built from the last 7 days only.
- **Hypothesis 2 (7.2 to 7.5):** The re-ranker's improvement comes from the user's personal history and product popularity rather than from query matching.
    - JD's search engine already returns products that match the query, so re-checking the query match should add little. Hypothesis 2 is supported if removing the personal-history features causes the largest drop in score and removing the query-match features only a small one.

**Which model runs which experiment:** The feature-family experiment, the direct test of Hypothesis 2, is run on both models: BST in 7.2 (models trained in Step 10 of `Test_BST_23_Features.ipynb` and scored here) and LightGBM in 7.3 and 7.4. The time-window experiment (7.1) and feature importance (7.5) are run on LightGBM only, for two different reasons.

- The time-window experiment needs three models trained from scratch on rebuilt feature files, one per window. Training a BST is far more expensive than training a LightGBM ranker, because it learns product embeddings and attention layers over every user's history sequence while LightGBM fits trees on the 23 features directly. The window experiment also does not depend on the model: both models see exactly the same 23 features, and the experiment changes only the history those features are built from, so a change in score reflects the value of the older history rather than anything about the ranker. Hence it was run on LightGBM, the cheaper of the two to retrain, and its conclusion about older history applies to the BST as well. BST training was reserved for the feature-family experiment in 7.2, where the model itself matters, since the question there is which features the BST depends on.

- Feature importance by gain exists only for tree models: it records how much each split on a feature reduced the training loss, and a neural network has no splits and therefore no equivalent number. Hence 7.5 reports gain importance for LightGBM only, and for the BST the family drops in 7.2 play the same role of showing which features the model depends on. The two are read together: 7.5 shows what the white-box model used, and 7.2 shows what changes the main model's score.

**Common settings for the LightGBM runs:** Each run trains on the same random quarter of the training sessions (`TRAIN_FRACTION`) and is scored on all validation sessions, so the numbers are below the full model's but comparable with each other. `EXP_FEATURES` and `LGB_PARAMS` are the 23 features and the "shallow" settings of `Jingxuan_LightGBM_23_Features.ipynb`, and early stopping after 100 rounds without improvement is the same rule as there. The BST runs in 7.2 also train on a quarter of the training sessions, with the settings of `Test_BST_23_Features.ipynb`.

The setup cell below runs first; 7.1 needs `Elysia_Window_Features.ipynb` to have been run.

In [15]:
import lightgbm as lgb

# ----- Final choices of Jingxuan_LightGBM_23_Features.ipynb -----
# If that notebook changes its feature list or settings, copy the new ones here and rerun Section 7.
EXP_FEATURES = [
    # personal history (Part3_Feature_Engineering / Aaron_All_Features_for_LightGBM)
    "exact_wid_last_seen_days", "shop_share", "shop_last_seen_days", "title_cos_max",
    "i2v_cos_mean", "last_i2v_cos", "shop_aff_slate_pct",
    # current-query match (ZhangNingJing_cleaned)
    "name_query_match", "brand_name_query_match", "cate_name_1_query_match", "cate_name_2_query_match",
    "cate_name_3_query_match", "cate_name_4_query_match", "query_match_count",
    # additional history / product / slate context (Aaron_All_Features_for_LightGBM)
    "brand_aff", "cate3_aff", "last_same_brand", "last_same_cate3",
    "qhist_wid_cnt", "title_cos_profile", "cate3_slate_share",
    # training-only popularity priors, out-of-fold (Aaron_All_Features_for_LightGBM)
    "item_ctr_train", "cate3_ctr_train",
]
# "Shallow" configuration chosen on validation NDCG@10 = 0.405. Same gains as the scoring code (click 1, cart 3, order 7)
LGB_PARAMS = dict(objective="lambdarank", metric="ndcg", eval_at=[10], label_gain=[0, 1, 3, 7],
                  num_leaves=15, min_data_in_leaf=150, learning_rate=0.05,
                  max_bin=255, feature_fraction=1.0, bagging_fraction=0.9, bagging_freq=1,
                  lambda_l2=1.0, verbosity=-1, seed=42, feature_fraction_seed=42, bagging_seed=42)
NUM_ROUNDS, EARLY_STOP = 1600, 100
TRAIN_FRACTION = 0.25      # experiments train on a quarter of the training sessions to save time

# Features flagged in the feature dictionary as close to JD's own ranking. The first two are never used;
# the others are allowed as product-level features.
RED_IN_EXCEL = ["item_impr_train", "i2v_cos_max_slate_pct", "i2v_cos_max", "i2v_cos_max_slate_z",
                "title_cos_max_slate_z", "title_cos_max_slate_pct", "title_cos_profile_slate_z",
                "title_cos_profile_slate_pct", "brand_ctr_train", "shop_aff_slate_z",
                "item_ctr_train_slate_pct", "cate3_slate_share", "qhist_wid_cnt"]
NEVER_USE = ["item_impr_train", "i2v_cos_max_slate_pct", "position"]
# --------------------------------------------------------------
missing = [f for f in EXP_FEATURES if f not in ALL_FEATURES + QUERY_FEATURES]
assert not missing, f"not in the feature files: {missing}"
assert not set(EXP_FEATURES) & set(NEVER_USE), "remove the never-use features"
print("flagged features that are used:", [f for f in EXP_FEATURES if f in RED_IN_EXCEL])


def train_model(tr, tr_group, va, va_group, feats):
    dtr = lgb.Dataset(tr[feats], tr["label"], group=tr_group, feature_name=list(feats), free_raw_data=True)
    dva = lgb.Dataset(va[feats], va["label"], group=va_group, reference=dtr)
    return lgb.train(LGB_PARAMS, dtr, num_boost_round=NUM_ROUNDS, valid_sets=[dva],
                     callbacks=[lgb.early_stopping(EARLY_STOP, verbose=False), lgb.log_evaluation(0)])


def run_experiment(name, feats, results_list, folder=FEAT_DIR):
    tr, tr_g = load_split("train", feats, folder, session_fraction=TRAIN_FRACTION)
    va, va_g = load_split("val", feats, folder)
    va = add_cold(va)
    model = train_model(tr, tr_g, va, va_g, feats)
    del tr; gc.collect()
    score = model.predict(va[feats], num_iteration=model.best_iteration)
    print(f"--- {name}: {len(feats)} features, {model.best_iteration} trees")
    add_result(results_list, name, "val", evaluate_ranking(to_pred(va, score)))
    del va; gc.collect()
    return model

flagged features that are used: ['qhist_wid_cnt', 'cate3_slate_share']


### 7.1 Time-window experiment (LightGBM)

Tests Hypothesis 1. Three LightGBM models with the same 23 features, settings and users, differing only in how much history the features were built from: **last 7 days**, **up to 90 days**, **full year** (the original feature files).

The features cannot be windowed by dropping columns, because most history features combine all past actions into one decayed number and none of the 23 is a window count. `Elysia_Window_Features.ipynb` therefore rebuilds every feature from history cut at 7 and 90 days and saves them to `processed/lgbm_features_window_7d/` and `..._90d/`; `WINDOW_METHOD = "rebuild"` reads those. (`"columns"` is a fallback that drops `_7_90d` and `_90_365d` columns, which does nothing for these 23 features.)

The "Full year" run uses the same data and settings as "All features" in 7.3, so the two scores should match.

In [ ]:
WINDOW_METHOD = "rebuild"

window_results = []
if WINDOW_METHOD == "rebuild":
    window_runs = {
        "Last 7 days only": PROCESSED / "lgbm_features_window_7d",
        "Up to 90 days":    PROCESSED / "lgbm_features_window_90d",
        "Full year":        FEAT_DIR,
    }
    for name, folder in window_runs.items():
        assert (folder / "train.parquet").exists(), f"{folder.name} missing: run Elysia_Window_Features.ipynb first"
        run_experiment(name, EXP_FEATURES, window_results, folder)
else:
    older = lambda f: f.endswith("_90_365d")
    mid = lambda f: f.endswith("_7_90d")
    window_runs = {
        "Last 7 days only": [f for f in EXP_FEATURES if not (older(f) or mid(f))],
        "Up to 90 days":    [f for f in EXP_FEATURES if not older(f)],
        "Full year":        EXP_FEATURES,
    }
    for name, feats in window_runs.items():
        run_experiment(name, feats, window_results)

window_table = results_table(window_results)
window_table

train: 34,180 sessions, 3,061,821 rows, 23 features
val: 17,117 sessions, 1,536,651 rows, 23 features
--- Last 7 days only: 23 features, 437 trees
Scored 17,117 users (1,986 with a purchase)
train: 34,180 sessions, 3,061,821 rows, 23 features
val: 17,117 sessions, 1,536,651 rows, 23 features
--- Up to 90 days: 23 features, 536 trees
Scored 17,117 users (1,986 with a purchase)
train: 34,180 sessions, 3,061,821 rows, 23 features
val: 17,117 sessions, 1,536,651 rows, 23 features
--- Full year: 23 features, 651 trees
Scored 17,117 users (1,986 with a purchase)


,NDCG@10,MRR,P@1,EngRecall@10,PurRecall@10,EngRecall@10 cold,EngRecall@10 warm
split,val,val,val,val,val,val,val
model,,,,,,,
Last 7 days only,0.3896,0.3926,0.2514,0.5789,0.6235,0.1883,0.6636
Up to 90 days,0.3994,0.4031,0.2599,0.5883,0.6368,0.1867,0.6762
Full year,0.4023,0.4062,0.2636,0.5907,0.6432,0.1885,0.6781


**Result:** Validation NDCG@10 rises from 0.390 with the last 7 days of history to 0.399 with 90 days and 0.402 with the full year. Hypothesis 1 is supported: older history improves the ranking, with most of the gain coming from extending the window from 7 to 90 days, and a smaller gain from the last 275 days.

### 7.2 Feature-family experiment on BST

Tests Hypothesis 2 on the main model. Step 10 of `Test_BST_23_Features.ipynb` trained twelve BSTs on a quarter of the training sessions: all features, then one feature family zeroed out of the context input at a time, then all personal-history families at once, each early-stopped on validation graded NDCG@10. The saved best-epoch validation predictions in `processed/bst_experiments/` are scored here with the shared scoring code, so this table reads like the LightGBM one in 7.3. The BST runs use a different random quarter from the LightGBM runs, so the drops are comparable across the two tables but the absolute numbers are not.

In [5]:
BST_EXP_DIR = PROCESSED / "bst_experiments"
bst_runs = pd.read_csv(BST_EXP_DIR / "bst_family_experiment_runs.csv")

val_cold = add_cold(load_split("val")[0])[["session_id", "wid", "is_cold"]].drop_duplicates(["session_id", "wid"])

bst_family_results = []
for _, r in bst_runs.iterrows():
    p = pd.read_parquet(BST_EXP_DIR / r["predictions"])
    p = p.merge(val_cold, on=["session_id", "wid"], how="left")
    p = p.rename(columns={"session_id": "user_id", "wid": "item_id"})
    print(f"--- {r['model']}")
    add_result(bst_family_results, r["model"], "val", evaluate_ranking(p, verbose=False))
    del p; gc.collect()

bst_family_table = pd.DataFrame(bst_family_results).set_index("model")[MAIN_COLUMNS]
bst_family_table["NDCG@10 drop"] = bst_family_table.loc["All features", "NDCG@10"] - bst_family_table["NDCG@10"]
bst_family_table.to_csv(RESULTS_DIR / "bst_family_experiment.csv")
bst_family_table.round(4)

val: 17,117 sessions, 1,536,651 rows, 0 features
--- All features
--- Without A: Exact product history
--- Without B: Brand / shop / category affinity
--- Without C: Title similarity to history
--- Without D: item2vec similarity
--- Without E: Similarity to the last action
--- Without F: History under the same query
--- Without G: Popularity priors (training users only)
--- Without H: Relative to the other candidates in the slate
--- Without Q: Current-query match
--- Without all personal history (A-F)


,NDCG@10,MRR,P@1,EngRecall@10,PurRecall@10,EngRecall@10 cold,EngRecall@10 warm,NDCG@10 drop
model,,,,,,,,
All features,0.3819,0.3891,0.2526,0.5600,0.6164,0.1653,0.6464,0.0000
Without A: Exact product history,0.3794,0.3851,0.2475,0.5587,0.6204,0.1704,0.6431,0.0025
Without B: Brand / shop / category affinity,0.3809,0.3883,0.2516,0.5585,0.6161,0.1687,0.6433,0.0010
Without C: Title similarity to history,0.3747,0.3816,0.2441,0.5517,0.6139,0.1647,0.6359,0.0071
Without D: item2vec similarity,0.3781,0.3849,0.2471,0.5559,0.6149,0.1676,0.6403,0.0037
Without E: Similarity to the last action,0.3794,0.3857,0.2476,0.5584,0.6132,0.1661,0.6437,0.0024
Without F: History under the same query,0.3810,0.3881,0.2503,0.5589,0.6149,0.1668,0.6443,0.0009
Without G: Popularity priors (training users only),0.3514,0.3577,0.2235,0.5283,0.5758,0.1639,0.6076,0.0304
Without H: Relative to the other candidates in the slate,0.3791,0.3857,0.2495,0.5568,0.6140,0.1629,0.6425,0.0028


**Result:** With all features the quarter-sample BST scores 0.382. Removing all personal-history features costs 0.057, removing the popularity family 0.030, and removing any single family at most 0.007; the query-match family costs 0.005. Hypothesis 2 is supported on the main model: personal history matters most, popularity second, and the query match least. Every run reached its best validation score after the first epoch and declined after that, consistent with the training-to-test gap in Section 6.

### 7.3 Feature-family experiment on LightGBM

Tests Hypothesis 2 on the white-box model. Trains with all `EXP_FEATURES`, then removes one family at a time (families from the feature dictionary, plus the query-match family Q). The larger the drop in validation NDCG@10, the more the model depends on that family.

Ten runs. Each finished run is saved to `4B_results/family_experiment_runs.csv` and the all-features model to `family_full_model.txt`, so a rerun skips runs already saved and only rebuilds the table. The table reads every run in that file, so it also shows the all-personal-history run from 7.4 once that has been saved.

In [17]:
families_present = sorted({FAMILY_OF[f] for f in EXP_FEATURES})
print("families in EXP_FEATURES:", {f: FAMILY_NAMES[f] for f in families_present})

FAM_FILE = RESULTS_DIR / "family_experiment_runs.csv"     # saved after every run
MODEL_FILE = RESULTS_DIR / "family_full_model.txt"        # the all-features model, for 7.5
done = pd.read_csv(FAM_FILE) if FAM_FILE.exists() else pd.DataFrame(columns=["model"])
family_results = done.to_dict("records")

runs = [("All features", EXP_FEATURES)] + [
    (f"Without {fam}: {FAMILY_NAMES[fam]}", [f for f in EXP_FEATURES if FAMILY_OF[f] != fam])
    for fam in families_present]

for name, feats in runs:
    if name in set(done["model"]):
        print("already done, skipping:", name)
        continue
    if not feats:
        continue
    model = run_experiment(name, feats, family_results)
    if name == "All features":
        model.save_model(str(MODEL_FILE))
    pd.DataFrame(family_results).to_csv(FAM_FILE, index=False)

full_model = lgb.Booster(model_file=str(MODEL_FILE))
family_table = pd.DataFrame(family_results).set_index("model")[MAIN_COLUMNS]
family_table["NDCG@10 drop"] = family_table.loc["All features", "NDCG@10"] - family_table["NDCG@10"]
family_table.round(4)

families in EXP_FEATURES: {'A': 'Exact product history', 'B': 'Brand / shop / category affinity', 'C': 'Title similarity to history', 'D': 'item2vec / term2vec similarity', 'E': 'Similarity to the last action', 'F': 'History under the same query', 'G': 'Popularity priors (training users only)', 'H': 'Relative to the other candidates in the slate', 'Q': 'Current-query match'}
already done, skipping: All features
already done, skipping: Without A: Exact product history
already done, skipping: Without B: Brand / shop / category affinity
already done, skipping: Without C: Title similarity to history
already done, skipping: Without D: item2vec / term2vec similarity
already done, skipping: Without E: Similarity to the last action
already done, skipping: Without F: History under the same query
already done, skipping: Without G: Popularity priors (training users only)
already done, skipping: Without H: Relative to the other candidates in the slate
already done, skipping: Without Q: Current-que

,NDCG@10,MRR,P@1,EngRecall@10,PurRecall@10,EngRecall@10 cold,EngRecall@10 warm,NDCG@10 drop
model,,,,,,,,
All features,0.4023,0.4062,0.2636,0.5907,0.6432,0.1885,0.6781,0.0000
Without A: Exact product history,0.4017,0.4054,0.2624,0.5912,0.6472,0.1894,0.6784,0.0006
Without B: Brand / shop / category affinity,0.4012,0.4047,0.2619,0.5904,0.6444,0.1884,0.6781,0.0011
Without C: Title similarity to history,0.3970,0.4021,0.2617,0.5833,0.6323,0.1733,0.6738,0.0053
Without D: item2vec / term2vec similarity,0.3987,0.4030,0.2621,0.5864,0.6318,0.1958,0.6704,0.0036
Without E: Similarity to the last action,0.4018,0.4051,0.2616,0.5914,0.6469,0.1882,0.6794,0.0005
Without F: History under the same query,0.4015,0.4054,0.2620,0.5904,0.6413,0.1888,0.6778,0.0007
Without G: Popularity priors (training users only),0.3503,0.3545,0.2207,0.5314,0.5790,0.1648,0.6098,0.0520
Without H: Relative to the other candidates in the slate,0.4002,0.4037,0.2604,0.5893,0.6318,0.1898,0.6766,0.0021


**Result:** Removing the popularity family costs LightGBM 0.052, the largest single-family drop. Removing any one personal-history family costs at most 0.005, because the other history families carry the same information. The query-match family costs about as little as any single history family, so re-checking the query match adds almost nothing on top of JD's own retrieval.

### 7.4 All personal-history features removed at once (LightGBM)

The personal-history families (A to F) overlap, so removing one at a time understates their value: the others compensate. This run removes all of them together (plus `shop_aff_slate_pct`, the slate version of shop affinity), leaving only query-match, popularity and slate-share features. The drop from "All features" is the value of personalisation as a whole, the direct test of Hypothesis 2. The run is appended to `family_experiment_runs.csv`; the 7.3 runs are not repeated.

In [18]:
# Remove ALL personal-history features at once (families A-F, plus the slate version of shop affinity)
FAM_FILE = RESULTS_DIR / "family_experiment_runs.csv"      # the file 7.3 saved its runs in
PERSONAL_FAMILIES = {"A", "B", "C", "D", "E", "F"}
no_personal = [f for f in EXP_FEATURES if FAMILY_OF[f] not in PERSONAL_FAMILIES and f != "shop_aff_slate_pct"]
print("features kept:", no_personal)

family_results = pd.read_csv(FAM_FILE).to_dict("records")
name = "Without all personal history (A-F)"
if name not in {r["model"] for r in family_results}:
    run_experiment(name, no_personal, family_results)
    pd.DataFrame(family_results).to_csv(FAM_FILE, index=False)

family_table = pd.DataFrame(family_results).set_index("model")[MAIN_COLUMNS]
family_table["NDCG@10 drop"] = family_table.loc["All features", "NDCG@10"] - family_table["NDCG@10"]
family_table.round(4)

features kept: ['name_query_match', 'brand_name_query_match', 'cate_name_1_query_match', 'cate_name_2_query_match', 'cate_name_3_query_match', 'cate_name_4_query_match', 'query_match_count', 'cate3_slate_share', 'item_ctr_train', 'cate3_ctr_train']


,NDCG@10,MRR,P@1,EngRecall@10,PurRecall@10,EngRecall@10 cold,EngRecall@10 warm,NDCG@10 drop
model,,,,,,,,
All features,0.4023,0.4062,0.2636,0.5907,0.6432,0.1885,0.6781,0.0000
Without A: Exact product history,0.4017,0.4054,0.2624,0.5912,0.6472,0.1894,0.6784,0.0006
Without B: Brand / shop / category affinity,0.4012,0.4047,0.2619,0.5904,0.6444,0.1884,0.6781,0.0011
Without C: Title similarity to history,0.3970,0.4021,0.2617,0.5833,0.6323,0.1733,0.6738,0.0053
Without D: item2vec / term2vec similarity,0.3987,0.4030,0.2621,0.5864,0.6318,0.1958,0.6704,0.0036
Without E: Similarity to the last action,0.4018,0.4051,0.2616,0.5914,0.6469,0.1882,0.6794,0.0005
Without F: History under the same query,0.4015,0.4054,0.2620,0.5904,0.6413,0.1888,0.6778,0.0007
Without G: Popularity priors (training users only),0.3503,0.3545,0.2207,0.5314,0.5790,0.1648,0.6098,0.0520
Without H: Relative to the other candidates in the slate,0.4002,0.4037,0.2604,0.5893,0.6318,0.1898,0.6766,0.0021


**Result:** Removing all personal-history features at once costs 0.072, more than the popularity family and more than any single family. Hypothesis 2 is supported on the white-box model as well: the re-ranker's value comes from the user's history first and popularity second, with the same order as the BST in 7.2.

### 7.5 Feature importance (LightGBM)

Gain importance from the all-features model of 7.3: how much each feature reduced the training loss across all trees, also summed by family because near-duplicate features (e.g. `shop_aff` and `shop_share`) share their importance. Gain shows what the model used; the family drops in 7.2 to 7.4 show what changes the score. Agreement between the two strengthens the conclusion on Hypothesis 2.

In [19]:
importance = pd.DataFrame({"feature": EXP_FEATURES,
                           "gain": full_model.feature_importance(importance_type="gain")})
importance["share"] = importance["gain"] / importance["gain"].sum()
importance["family"] = importance["feature"].map(FAMILY_OF).map(FAMILY_NAMES)
importance = importance.sort_values("share", ascending=False)

print(importance.groupby("family")["share"].sum().sort_values(ascending=False).round(3))
importance.head(15).round(4)

family
item2vec / term2vec similarity                   0.396
Popularity priors (training users only)          0.239
Exact product history                            0.137
Title similarity to history                      0.114
Brand / shop / category affinity                 0.052
Relative to the other candidates in the slate    0.025
Current-query match                              0.021
Similarity to the last action                    0.015
History under the same query                     0.002
Name: share, dtype: float64


,feature,gain,share,family
4,i2v_cos_mean,407144.2078,0.3956,item2vec / term2vec similarity
21,item_ctr_train,237403.5594,0.2307,Popularity priors (training users only)
0,exact_wid_last_seen_days,140597.4526,0.1366,Exact product history
3,title_cos_max,107753.5523,0.1047,Title similarity to history
1,shop_share,22997.1747,0.0223,Brand / shop / category affinity
7,name_query_match,18598.9062,0.0181,Current-query match
20,cate3_slate_share,18478.3281,0.0180,Relative to the other candidates in the slate
2,shop_last_seen_days,18065.0853,0.0176,Brand / shop / category affinity
5,last_i2v_cos,14355.5475,0.0139,Similarity to the last action
19,title_cos_profile,9109.0230,0.0089,Title similarity to history


**Result:** Gain importance agrees with the family drops: the item2vec similarity features take about 40% of total gain, popularity 24%, the exact-product-seen features 14% and title similarity 11%, while the query-match features are near the bottom. What the model used and what changes its score point to the same answer, personal history and popularity rather than query matching.

## 8. Save everything

Writes every result present in this session to `model_artifacts/4B_results/`: the results table from Sections 5 and 6, the time-window experiment (7.1), the LightGBM feature-family experiment (7.3 and 7.4, one table) and the feature importance (7.5). The BST feature-family table was already written by 7.2. Anything not yet run is skipped, so this cell can be run after any section.

In [20]:
if "all_results" in globals() and all_results:
    pd.DataFrame(all_results).to_csv(RESULTS_DIR / "all_results.csv", index=False)
    results_table(all_results).to_csv(RESULTS_DIR / "main_results_table.csv")
if "window_results" in globals() and window_results:
    pd.DataFrame(window_results).to_csv(RESULTS_DIR / "window_experiment.csv", index=False)
if "family_results" in globals() and family_results:
    family_table.to_csv(RESULTS_DIR / "family_experiment.csv")
if "importance" in globals():
    importance.to_csv(RESULTS_DIR / "feature_importance.csv", index=False)
print("saved to", RESULTS_DIR)

saved to /content/drive/MyDrive/BT4222 Final/JDSearch/model_artifacts/4B_results
